<a href="https://colab.research.google.com/github/disCarlosBustos/aprendizaje_supervisado/blob/main/5_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ============================================================
# 2.6 PIPELINES - APRENDIZAJE SUPERVISADO
# Proyecto Titanic
# ============================================================

# ------------------------------------------------------------
# 1. Importar paquetes
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import pickle

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

from sklearn.metrics import accuracy_score


# ------------------------------------------------------------
# 2. Carga de datos
# ------------------------------------------------------------

df = pd.read_csv('./data/titanic.csv')

print("Primeras filas del dataset:")
display(df.head())

print("Columnas originales:")
print(df.columns)


# ------------------------------------------------------------
# 3. Limpieza mínima para usar el dataset original
# ------------------------------------------------------------

# Este repositorio trae titanic.csv, no titanic_clean.csv.
# Por eso hacemos aquí una limpieza básica antes de crear el Pipeline.

# Conservar únicamente las columnas necesarias para el modelo
columnas_modelo = [
    'Survived',
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Embarked'
]

df = df[columnas_modelo].copy()

# Rellenar valores faltantes
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print("Dataset después de limpieza mínima:")
display(df.head())

print("Valores faltantes después de limpieza:")
print(df.isnull().sum())


# ------------------------------------------------------------
# 4. Preprocesamiento con ColumnTransformer
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'), ['Sex', 'Embarked']),
        ('age', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'
)


# ------------------------------------------------------------
# 5. Definición de modelos e hiperparámetros
# ------------------------------------------------------------

modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear'],
            'model__max_iter': [500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'rbf'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}


# ------------------------------------------------------------
# 6. División de datos
# ------------------------------------------------------------

X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=100
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)


# ------------------------------------------------------------
# 7. Variables auxiliares
# ------------------------------------------------------------

puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}


# ------------------------------------------------------------
# 8. Ciclo GridSearchCV con Pipeline
# ------------------------------------------------------------

for nombre, info_modelo in modelos.items():
    print("Entrenando:", nombre)

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),
        ('model', info_modelo['modelo'])
    ])

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train)

    y_pred = grid_search.predict(X_test)

    precision = accuracy_score(y_test, y_pred)

    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_

    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

print("Entrenamiento terminado.")


# ------------------------------------------------------------
# 9. Mostrar resultados
# ------------------------------------------------------------

metricas = pd.DataFrame(puntajes_modelos).sort_values(
    'Precisión',
    ascending=False
)

print("Rendimiento de los modelos de clasificación")
display(metricas.round(4))

print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.4f}")


# ------------------------------------------------------------
# 10. Guardar el mejor Pipeline
# ------------------------------------------------------------

with open('pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print("Pipeline guardado correctamente como pipeline.pkl")


# ------------------------------------------------------------
# 11. Prueba de predicción con datos sin transformar
# ------------------------------------------------------------

ejemplo = pd.DataFrame([{
    "Pclass": 2,
    "Sex": "male",
    "Age": 46,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.2500,
    "Embarked": "C"
}])

prediccion = mejor_estimador.predict(ejemplo)

print("Predicción con JSON sin transformar:")
print(ejemplo)
print("Resultado Survived:", int(prediccion[0]))


Primeras filas del dataset:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Columnas originales:
Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')
Dataset después de limpieza mínima:


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


Valores faltantes después de limpieza:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Tamaño de X_train: (712, 7)
Tamaño de X_test: (179, 7)
Entrenando: Regresión Logística
Entrenando: Clasificador de Vectores de Soporte
Entrenando: Clasificador de Árbol de Decisión
Entrenando: Clasificador de Bosques Aleatorios
Entrenando: Clasificador de Gradient Boosting
Entrenando: Clasificador AdaBoost
Entrenando: Clasificador K-Nearest Neighbors
Entrenando: GaussianNB
Entrenando: Clasificador Naive Bayes
Entrenamiento terminado.
Rendimiento de los modelos de clasificación


,Modelo,Precisión
6,Clasificador K-Nearest Neighbors,0.8324
0,Regresión Logística,0.8101
1,Clasificador de Vectores de Soporte,0.8101
5,Clasificador AdaBoost,0.8101
2,Clasificador de Árbol de Decisión,0.8045
3,Clasificador de Bosques Aleatorios,0.8045
4,Clasificador de Gradient Boosting,0.7989
7,GaussianNB,0.7877
8,Clasificador Naive Bayes,0.7765


---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.8324
Pipeline guardado correctamente como pipeline.pkl
Predicción con JSON sin transformar:
   Pclass   Sex  Age  SibSp  Parch  Fare Embarked
0       2  male   46      0      0  7.25        C
Resultado Survived: 0
